# Chapter 12 explore: Detecting Drift Across Model Versions

Interactive companion to `code/chapter_12/detect_model_drift.py`. Runs Chapter 11's evaluation harness across Chapter 8's 3 real epoch checkpoints and compares them pairwise, looking for metrics that disagree on direction. Needs Chapter 8's checkpoint to exist first (`python code/chapter_08/finetune_at_scale.py`, ~30 min).

In [1]:
import sys
sys.path.insert(0, "../code/chapter_01")
sys.path.insert(0, "../code/chapter_02")
sys.path.insert(0, "../code/chapter_06")
sys.path.insert(0, "../code/chapter_07")
sys.path.insert(0, "../code/chapter_09")
sys.path.insert(0, "../code/chapter_10")
sys.path.insert(0, "../code/chapter_11")
sys.path.insert(0, "../code/chapter_12")

from eval_finetuned_model import build_held_out_eval_set
from detect_model_drift import CHECKPOINTS_DIR, load_version, summarize_version, compare_versions

eval_set = build_held_out_eval_set()
runs = sorted(CHECKPOINTS_DIR.glob("run_*"))
checkpoints = sorted(runs[-1].glob("checkpoint_*"), key=lambda p: int(p.name.split("_")[1]))
print(f"{len(checkpoints)} real checkpoints found")

3 real checkpoints found


Summarize every checkpoint, then compare each pair for direction agreement or disagreement.

In [2]:
summaries = {}
for checkpoint_dir in checkpoints:
    model, tokenizer = load_version(checkpoint_dir)
    summaries[checkpoint_dir.name] = summarize_version(model, tokenizer, eval_set)
    print(checkpoint_dir.name, summaries[checkpoint_dir.name])

print()
names = list(summaries)
for before_name, after_name in zip(names, names[1:]):
    print(f"{before_name} -> {after_name}: {compare_versions(summaries[before_name], summaries[after_name])}")

`torch_dtype` is deprecated! Use `dtype` instead!


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


checkpoint_1 {'exact_match': 0, 'avg_overlap': 0.3541666666666667, 'perplexity': 27.985743752445202}


checkpoint_2 {'exact_match': 0, 'avg_overlap': 0.16666666666666666, 'perplexity': 25.399341287801796}


checkpoint_3 {'exact_match': 0, 'avg_overlap': 0.16666666666666666, 'perplexity': 25.033626962281886}

checkpoint_1 -> checkpoint_2: {'exact_match': 'unchanged', 'avg_overlap': 'regressed', 'perplexity': 'improved'}
checkpoint_2 -> checkpoint_3: {'exact_match': 'unchanged', 'avg_overlap': 'unchanged', 'perplexity': 'improved'}
